In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import psycopg2
from minio import Minio

spark = SparkSession.builder \
        .appName("Brazilian-Ecommerce") \
        .master("spark://spark-master:7077") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
        .config("spark.executor.memory", "1g") \
        .config("spark.driver.memory", "1g") \
        .getOrCreate()

# set log level
spark.sparkContext.setLogLevel("WARN")

print("✅ Spark Session created successfully")
print(f"📊 Spark Version: {spark.version}")
print(f"🔧 Application Name: {spark.sparkContext.appName}")
print(f"🎯 Master URL: {spark.sparkContext.master}")

✅ Spark Session created successfully
📊 Spark Version: 3.5.0
🔧 Application Name: Brazilian-Ecommerce
🎯 Master URL: spark://spark-master:7077


In [9]:
import os 
from minio import Minio
from minio.error import S3Error


def upload_data_to_minio():
    minio_client = Minio(
        "minio:9000",
        access_key="minio_admin",
        secret_key="minio_secure_pwd",
        secure=False
    )

    bucket_name = "raw-data"
    object_name = "/home/jovyan/work/data/raw/"
    try:
        # check bucket có tồn tại ko nếu ko thì tạo 
        if not minio_client.bucket_exists(bucket_name):
            minio_client.make_bucket(bucket_name)


        # lay cac file .csv va upload len minio
        print(f"Bắt đầu upload dữ liệu từ {object_name} lên MinIO...")
       
        for file_name in os.listdir(object_name):
            if file_name.endswith(".csv"):
                file_path = os.path.join(object_name, file_name)

            #start upload data into minio
                minio_client.fput_object(
                    bucket_name, 
                    file_name, 
                    file_path
                )
        
        print("\n--- Hoàn thành Ingestion! Dữ liệu đã sẵn sàng trên Data Lake. ---")
    except S3Error as e:
         print(f"Lỗi khi tương tác với MinIO: {e}")

upload_data_to_minio()

Bắt đầu upload dữ liệu từ /home/jovyan/work/data/raw/ lên MinIO...
['.gitkeep', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']

--- Hoàn thành Ingestion! Dữ liệu đã sẵn sàng trên Data Lake. ---
